# 01 — Google Colab: data acquisition, verification & Kaggle publishing

**Fire Harmonizer — MODIS–VIIRS physics-informed harmonization engine (NASA Space Apps 2026)**

This notebook is the *Environment 1 stager* of the project (readme §2.2 / §3.1). It
prepares **one self-contained, offline-ready execution package** and publishes it as a
**Kaggle dataset** that `02_kaggle_harmonization.ipynb` runs from — offline.

| Responsibility | Where it lives |
|---|---|
| Data source discovery, failover, download, HDF4 extraction, polygon clipping, Science-over-NRT dedup, outage calendar, coverage audit | **the codebase** — `scripts/run_env1_stage.py` → `src/ingest/download_data.py`, `src/ingest/normalize_dedup.py` |
| Offline wheel bundle for the Kaggle runtime | **the codebase** — `src/ingest/build_wheels.sh` |
| Cloning the exact repository commit, inspecting its entry points, installing Env 1 dependencies, verifying files, checksums, packaging, Kaggle authentication & publishing | **this notebook** (orchestration only) |

The notebook never re-implements acquisition or processing logic: every data operation is
a call into the repository it clones. If you change the codebase, correct a dataset, or
re-acquire something, re-run this notebook — it publishes a **new version** of the same
Kaggle dataset, and the Kaggle notebook picks it up with **no edits** (it auto-discovers the
package through `package_manifest.json`).

### Verified data sources (checked 2026-11-14 from an internet-connected host)

| Product | Official source | Status |
|---|---|---|
| `MCD14ML` (MODIS C6.1), `VNP14IMGML` / `VJ114IMGML` / `VJ214IMGML` (VIIRS) monthly active fires | UMD fire archive `sftp://fuoco.geog.umd.edu` (login `fire`/`burnt`, FTP port 21 fallback) | primary; probed at run time (raw TCP 22/21) |
| `MOD14C8Q` / `MYD14C8Q` (8-day 0.25° cloud CMG), `MOD14CMQ` / `MYD14CMQ` (monthly fallback) | UMD fire archive only | **NOT in NASA CMR** (verified: CMR has no such collection) — UMD is the only source |
| `MCD64CMQ` (0.25° monthly burned area, validation) | UMD fire archive only | **NOT in NASA CMR** — validation-only product |
| All MODIS + VIIRS FIRMS detections 2000→today (active-fire failover) | Source Cooperative Portolan GeoParquet mirror `https://data.source.coop/portolan-mirrors/firms-catalog/` | verified HTTP 200; year partitions 4 MB (2000) … 917 MB (2026) |
| Trailing 2026 NRT gap-fill | NASA FIRMS REST API `https://firms.modaps.eosdis.nasa.gov/api/...` | verified live (HTTP 401 without key) — needs a free **MAP_KEY** |
| Admin-0 country polygons | Natural Earth 50m GeoJSON (GitHub raw) → GeoBoundaries fallback | verified HTTP 200 |

The notebook re-probes every source with the codebase's `probe_sources()` before downloading.

### Before you run
1. A **Kaggle account** and its API token file `kaggle.json` (Kaggle → *Settings* → *API* → *Create New Token*).
   Provide it either as Colab **Secrets** `KAGGLE_USERNAME` + `KAGGLE_KEY` (🔑 icon in the left sidebar), or by uploading `kaggle.json` when prompted.
   If your Kaggle settings page gives a single **API token** instead, store it as secret `KAGGLE_API_TOKEN` (plus `KAGGLE_USERNAME`).
2. Optional: a free **FIRMS MAP_KEY** (https://firms.modaps.eosdis.nasa.gov/api/map_key/) as Colab secret `FIRMS_MAP_KEY` — needed only for the trailing-2026 NRT gap-fill.
3. Disk: the six-country package is ≈ 1–3 GB, but downloads stream global monthly files. Mount Google Drive (`USE_GOOGLE_DRIVE = True`) to make acquisition resumable across Colab sessions.

Run the cells top to bottom (*Runtime → Run all*). Every stage is idempotent and resumable.

## 1. Configuration

All tunables live here. Every value can also be overridden by an environment variable of the
same name prefixed with `FH_` (useful for headless runs).

In [ ]:
import os, sys, json, re, time, shutil, hashlib, subprocess, platform, datetime as dt
from pathlib import Path


def _env(name, default, cast=str):
    v = os.environ.get("FH_" + name)
    if v is None:
        return default
    if cast is bool:
        return v.strip().lower() in ("1", "true", "yes", "on")
    if cast is list:
        return [int(x) for x in re.split(r"[,\s]+", v.strip()) if x]
    return cast(v)


CONFIG = {
    # --- codebase -----------------------------------------------------------
    "REPO_URL": _env("REPO_URL", "https://github.com/Vtheonly/Alalah_nasa.git"),
    "REPO_REF": _env("REPO_REF", "main"),            # branch, tag or commit
    # --- storage ------------------------------------------------------------
    "USE_GOOGLE_DRIVE": _env("USE_GOOGLE_DRIVE", False, bool),
    "DRIVE_SUBDIR": _env("DRIVE_SUBDIR", "fire_harmonizer"),
    "WORK_ROOT": _env("WORK_ROOT", "/content/fire_harmonizer"),
    # --- acquisition (executed by the codebase) -------------------------------
    "RUN_ACQUISITION": _env("RUN_ACQUISITION", True, bool),
    "YEARS": _env("YEARS", list(range(2000, 2027)), list),
    "KEEP_ORIGINALS": _env("KEEP_ORIGINALS", "ancillary"),   # none | ancillary | all
    "STAGE_DIR": _env("STAGE_DIR", ""),     # reuse an existing Env 1 base dir
    "MIN_FREE_GB": _env("MIN_FREE_GB", 40.0, float),
    # --- offline wheels for the Kaggle runtime --------------------------------
    "TARGET_ENV2_PYTHON": _env("TARGET_ENV2_PYTHON", "3.13"),
    # --- packaging / publishing ----------------------------------------------
    "PUBLISH_ORIGINALS": _env("PUBLISH_ORIGINALS", False, bool),
    "PUBLISH_TO_KAGGLE": _env("PUBLISH_TO_KAGGLE", True, bool),
    "KAGGLE_DATASET_SLUG": _env("KAGGLE_DATASET_SLUG", "fire-harmonizer-env2-package"),
    "KAGGLE_DATASET_TITLE": _env("KAGGLE_DATASET_TITLE",
                                 "Fire Harmonizer Env2 Package MODIS VIIRS"),
    "KAGGLE_PUBLIC": _env("KAGGLE_PUBLIC", False, bool),
    "VERSION_NOTES": _env("VERSION_NOTES", ""),
    "PUBLISH_TIMEOUT_MIN": _env("PUBLISH_TIMEOUT_MIN", 90.0, float),
}

# Package contract shared with 02_kaggle_harmonization.ipynb -- change both
# notebooks together if you ever change these names.
PACKAGE_SCHEMA_VERSION = 1
MANIFEST_NAME = "package_manifest.json"
LAYOUT = {"codebase": "codebase", "staged_data": "staged-data",
          "wheels": "offline-wheels", "originals": "originals"}

assert CONFIG["KEEP_ORIGINALS"] in ("none", "ancillary", "all"), CONFIG["KEEP_ORIGINALS"]
print(json.dumps(CONFIG, indent=2, default=str))

## 2. Runtime, storage and helpers

Detects Colab, optionally mounts Google Drive (persistent, resumable work area), creates the
work tree, and defines small orchestration helpers (logged subprocess runner, checksums).

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if CONFIG["USE_GOOGLE_DRIVE"] and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive") / CONFIG["DRIVE_SUBDIR"]
else:
    WORK = Path(CONFIG["WORK_ROOT"])

REPO_DIR = WORK / "repo"
STAGE_DIR = Path(CONFIG["STAGE_DIR"]) if CONFIG["STAGE_DIR"] else WORK / "stage"
ORIGINALS_DIR = WORK / "originals"
PACKAGE_DIR = WORK / "kaggle_package"
LOG_DIR = WORK / "logs"
for d in (WORK, STAGE_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"Colab: {IN_COLAB} | Python {platform.python_version()} | work dir: {WORK}")
print(f"stage dir: {STAGE_DIR}\nrun id: {RUN_ID}")
_du = shutil.disk_usage(WORK)
print(f"disk free at work dir: {_du.free / 2**30:.1f} GiB of {_du.total / 2**30:.1f} GiB")


def human(n):
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if abs(n) < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PiB"


def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def run(cmd, log_name, cwd=None, env=None, check=True, echo_every=1):
    # Run a command, stream its output live and tee it to logs/<log_name>.log.
    log_path = LOG_DIR / f"{RUN_ID}_{log_name}.log"
    full_env = dict(os.environ)
    full_env.update(env or {})
    t0 = time.time()
    print(f"$ {' '.join(map(str, cmd))}\n  (log: {log_path})")
    with open(log_path, "w") as log:
        proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, env=full_env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        tail = []
        for i, line in enumerate(proc.stdout):
            log.write(line)
            tail = (tail + [line])[-60:]
            if i % echo_every == 0:
                print(line, end="")
        rc = proc.wait()
    dur = time.time() - t0
    print(f"  -> exit {rc} in {dur:.0f}s")
    if check and rc != 0:
        raise RuntimeError(f"{log_name} failed (exit {rc}); last lines:\n" + "".join(tail))
    return {"rc": rc, "seconds": dur, "log": str(log_path)}

## 3. Acquire the project codebase (exact commit)

Clones the repository (or updates an existing clone on Drive), checks out `REPO_REF`, and
records the exact commit. That commit is what gets packaged — data and code always travel
together.

In [ ]:
def git(*args, check=True):
    out = subprocess.run(["git", "-C", str(REPO_DIR), *args], capture_output=True, text=True)
    if check and out.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed: {out.stderr.strip()}")
    return out.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    git("remote", "set-url", "origin", CONFIG["REPO_URL"])
    git("fetch", "--tags", "origin", CONFIG["REPO_REF"])
    git("checkout", "--force", "--detach", "FETCH_HEAD")
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--no-single-branch", CONFIG["REPO_URL"], str(REPO_DIR)],
                   check=True)
    git("fetch", "--tags", "origin", CONFIG["REPO_REF"])
    git("checkout", "--force", "--detach", "FETCH_HEAD")

CODEBASE = {
    "repo_url": CONFIG["REPO_URL"],
    "ref": CONFIG["REPO_REF"],
    "commit": git("rev-parse", "HEAD"),
    "commit_date": git("show", "-s", "--format=%cI", "HEAD"),
    "commit_subject": git("show", "-s", "--format=%s", "HEAD"),
    "dirty": bool(git("status", "--porcelain")),
}
assert not CODEBASE["dirty"], "the clone has local modifications; refusing to package it"
print(json.dumps(CODEBASE, indent=2))

## 4. Inspect the codebase: entry points, layout contract, dependencies

Nothing here is assumed: entry points come from the `Makefile` and `scripts/`, the data layout
from `src/config/paths.py` (`DataPaths`), dependency groups from `pyproject.toml`, and the
acquisition API is checked by parsing `src/ingest/download_data.py`. The result is saved as
`codebase_inspection.json` and embedded in the package manifest.

In [ ]:
import ast

try:
    import tomllib
except ImportError:          # Python < 3.11
    import tomli as tomllib

pyproject = tomllib.loads((REPO_DIR / "pyproject.toml").read_text())
extras = pyproject.get("project", {}).get("optional-dependencies", {})

make_targets = {}
mk = (REPO_DIR / "Makefile").read_text()
for m in re.finditer(r"^([A-Za-z0-9_-]+):[^\n]*\n((?:\t[^\n]*\n?)+)", mk, re.M):
    make_targets[m.group(1)] = [l.strip() for l in m.group(2).splitlines() if l.strip()]

scripts = {}
for p in sorted((REPO_DIR / "scripts").glob("*.py")):
    doc = ast.get_docstring(ast.parse(p.read_text())) or ""
    scripts[p.name] = doc.splitlines()[0] if doc else ""


def module_functions(rel):
    tree = ast.parse((REPO_DIR / rel).read_text())
    return sorted(n.name for n in tree.body if isinstance(n, ast.FunctionDef))


acq_api = module_functions("src/ingest/download_data.py")
norm_api = module_functions("src/ingest/normalize_dedup.py")

# DataPaths is dependency-free: resolve the Env 1 layout for our stage dir.
sys.path.insert(0, str(REPO_DIR))
for mod in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[mod]
from src.config.paths import DataPaths      # noqa: E402
from src.config import settings as S        # noqa: E402

stage_paths = DataPaths.from_env(str(STAGE_DIR))
layout_env1 = {k: str(getattr(stage_paths, k)) for k in (
    "raw_stage_dir", "active_fires_dir", "cloud_parquet", "burned_area_parquet",
    "country_polygons_parquet", "wheels_dir", "provenance_manifest")}

REQUIRED_ENTRYPOINTS = ["run_env1_stage.py", "run_env2_cpu_prep.py",
                        "run_env2_gpu_solve.py", "run_env2_validate.py"]
missing = [s for s in REQUIRED_ENTRYPOINTS if s not in scripts]
missing += [f for f in ("download_all", "probe_sources") if f not in acq_api]
missing += [f for f in ("run_normalization",) if f not in norm_api]
if not (REPO_DIR / "src/ingest/build_wheels.sh").exists():
    missing.append("src/ingest/build_wheels.sh")
if missing:
    raise RuntimeError(f"codebase lacks required entry points: {missing}")

INSPECTION = {
    "make_targets": make_targets,
    "scripts": scripts,
    "dependency_groups": extras,
    "requires_python": pyproject["project"].get("requires-python"),
    "acquisition_api": [f for f in acq_api if not f.startswith("_")],
    "env1_layout": layout_env1,
    "env2_expected_inputs": ["active_fires/", "cloud_visibility_025deg.parquet",
                             "country_polygons.parquet",
                             "burned_area_val_025deg.parquet (validation, optional)"],
    "spec": {"date_start": S.DATE_START, "date_end": S.DATE_END, "t_days": S.T_DAYS,
             "countries": list(S.TARGET_COUNTRY_ISOS)},
}
(WORK / "codebase_inspection.json").write_text(json.dumps(INSPECTION, indent=2))
print("Makefile targets:", {k: v[0] for k, v in make_targets.items()})
print("Entry-point scripts:", json.dumps(scripts, indent=1))
print("Dependency groups:", {k: len(v) for k, v in extras.items()})
print("Env 1 layout (DataPaths):", json.dumps(layout_env1, indent=1))

## 5. Install the Environment 1 dependencies

Installs exactly the `ingest` dependency group declared in `pyproject.toml` (plus the packages
the acquisition modules import). `pyhdf` ships manylinux wheels for CPython 3.9–3.14; if a
wheel is unavailable the HDF4 headers are installed with `apt` and it is built from source.

In [ ]:
ingest_reqs = list(extras.get("ingest", []))
for extra_pkg in ("pandas", "numpy", "scipy", "requests"):
    if extra_pkg not in ingest_reqs:
        ingest_reqs.append(extra_pkg)

r = run([sys.executable, "-m", "pip", "install", "-q", *ingest_reqs], "pip_ingest", check=False)
if r["rc"] != 0:
    print("pip failed (likely pyhdf without a wheel) -> installing libhdf4-dev and retrying")
    run(["bash", "-lc", "apt-get -qq update && apt-get -qq install -y libhdf4-dev"], "apt_hdf4")
    run([sys.executable, "-m", "pip", "install", "-q", *ingest_reqs], "pip_ingest_retry",
        env={"INCLUDE_DIRS": "/usr/include/hdf",
             "LIBRARY_DIRS": "/usr/lib/x86_64-linux-gnu/hdf"})

import importlib
import_names = {"beautifulsoup4": "bs4", "pyarrow": "pyarrow"}
versions = {}
for req in ingest_reqs:
    name = re.split(r"[<>=\[ ]", req)[0]
    mod = importlib.import_module(import_names.get(name, name))
    versions[name] = getattr(mod, "__version__", "?")
print(json.dumps(versions, indent=1))

# entry-point smoke test now that dependencies are present
_ = run([sys.executable, "scripts/run_env1_stage.py", "--help"], "env1_help", cwd=REPO_DIR,
        echo_every=10**9)

## 6. Data-source credentials

* `FIRMS_MAP_KEY` (optional) — trailing-2026 NRT gap-fill through the FIRMS API. Without it
  the codebase skips the NRT step and the package still contains the full Science record.
* `EARTHDATA_TOKEN` (optional) — only used by the codebase's NASA CMR fallback.

Values are read from Colab Secrets (or a hidden prompt) and passed to the codebase through
the process environment only. They are never printed, logged or packaged.

In [ ]:
from getpass import getpass


def secret(name, prompt=True):
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v:
                return v
        except Exception:
            pass
    if prompt and sys.stdin is not None and sys.stdin.isatty() or (prompt and IN_COLAB):
        v = getpass(f"{name} (leave empty to skip): ").strip()
        return v or None
    return None


DATA_SECRETS = {}
for _name in ("FIRMS_MAP_KEY", "EARTHDATA_TOKEN"):
    _v = secret(_name, prompt=CONFIG["RUN_ACQUISITION"])
    if _v:
        DATA_SECRETS[_name] = _v
print("data-source credentials provided:", sorted(DATA_SECRETS) or "none")

## 7. Probe every data source (codebase `probe_sources`)

Runs `scripts/run_env1_stage.py --probe-only`, i.e. the readme's "Minute 15" connectivity
check implemented in `src/ingest/download_data.py`. The acquisition cannot succeed without:

* **active fires**: UMD (SFTP 22 or FTP 21) **or** the Portolan mirror, and
* **cloud grids**: UMD only (they are not distributed through NASA CMR).

In [ ]:
PROBE = None
if CONFIG["RUN_ACQUISITION"]:
    _p = subprocess.run([sys.executable, "scripts/run_env1_stage.py", "--base-dir",
                         str(STAGE_DIR), "--probe-only"], cwd=REPO_DIR, capture_output=True,
                        text=True, env={**os.environ, **DATA_SECRETS})
    if _p.returncode != 0:
        raise RuntimeError(_p.stderr[-3000:])
    PROBE = json.loads(_p.stdout[_p.stdout.index("["):])
    for row in PROBE:
        print(f"{'OK  ' if row['reachable'] else 'FAIL'} {row['source']:<70} {row['detail']}")
    up = {row["endpoint"]: row["reachable"] for row in PROBE}
    umd = any(v for k, v in up.items() if k.startswith("fuoco.geog.umd.edu"))
    portolan = any(v for k, v in up.items() if "portolan" in k)
    if not (umd or portolan):
        raise RuntimeError("No active-fire source reachable (UMD and Portolan both down).")
    if not umd:
        raise RuntimeError(
            "UMD fire archive unreachable on ports 22/21: the MOD14C8Q/MYD14C8Q/Mx14CMQ cloud "
            "grids are distributed ONLY there (not in NASA CMR), and the codebase refuses to "
            "fabricate cloud fractions. Retry later or from another network.")
    if "FIRMS_MAP_KEY" not in DATA_SECRETS:
        print("NOTE: no FIRMS_MAP_KEY -> the trailing-2026 NRT gap-fill will be skipped.")
else:
    print("RUN_ACQUISITION=False: re-packaging the existing stage dir", STAGE_DIR)

## 8. Run the acquisition — the codebase's Environment 1 stager

`scripts/run_env1_stage.py` executes the full readme §6 pipeline: UMD SFTP→FTP discovery and
download (one file at a time), Portolan failover, FIRMS NRT gap-fill, CMG/MCD64CMQ HDF4
extraction, Natural Earth polygon clipping, Science-over-NRT dedup, `platform_active_day`,
and the coverage audit. Each original file's size + SHA-256 + remote path is written to
`data_provenance_manifest.json`.

It is resumable: staged monthly chunks, staged Portolan years and (with `KEEP_ORIGINALS`)
original files already on disk are reused on re-runs.

In [ ]:
ACQ = None
if CONFIG["RUN_ACQUISITION"]:
    free_gb = shutil.disk_usage(WORK).free / 2**30
    if free_gb < CONFIG["MIN_FREE_GB"]:
        raise RuntimeError(f"only {free_gb:.0f} GiB free at {WORK}; need "
                           f">= {CONFIG['MIN_FREE_GB']} GiB (mount Drive or free space)")
    cmd = [sys.executable, "scripts/run_env1_stage.py", "--base-dir", str(STAGE_DIR),
           "--years", *map(str, CONFIG["YEARS"]), "--skip-wheels"]
    if CONFIG["KEEP_ORIGINALS"] != "none":
        cmd += ["--originals-dir", str(ORIGINALS_DIR),
                "--keep-originals", CONFIG["KEEP_ORIGINALS"]]
    ACQ = run(cmd, "env1_stage", cwd=REPO_DIR, env=DATA_SECRETS)
else:
    print("skipped (RUN_ACQUISITION=False)")

## 9. Build the offline wheel bundle for the Kaggle runtime (codebase `build_wheels.sh`)

The Kaggle notebook has no internet, so every package missing from the Kaggle image must be
in the dataset. `build_wheels.sh` downloads `duckdb`, `polars`, `h3` wheels for the
**Kaggle** interpreter (`TARGET_ENV2_PYTHON`, 3.13 on the 4× L4 image) — not for Colab's own
Python — and verifies their ABI tags. The Kaggle notebook installs only what is missing
(on the current image: `h3`).

In [ ]:
WHEELS_DIR = Path(stage_paths.wheels_dir)
WHEELS_DIR.mkdir(parents=True, exist_ok=True)
run(["bash", "src/ingest/build_wheels.sh", str(WHEELS_DIR), CONFIG["TARGET_ENV2_PYTHON"]],
    "build_wheels", cwd=REPO_DIR)
wheels = sorted(p.name for p in WHEELS_DIR.glob("*.whl"))
py_tag = "cp" + CONFIG["TARGET_ENV2_PYTHON"].replace(".", "")
assert any(w.startswith("h3-") and py_tag in w for w in wheels), f"no {py_tag} h3 wheel: {wheels}"
print("\n".join(wheels))

## 10. Verify the staged data

Checks the codebase's own provenance manifest and coverage audit, then verifies every staged
file independently: required files present, every Parquet footer readable (detects truncated
or corrupted downloads), the six country polygons present, cloud-grid coverage, and the
SHA-256 of every preserved original against the manifest.

In [ ]:
import pyarrow.parquet as pq

problems, warnings_ = [], []
STAGED = Path(stage_paths.raw_stage_dir).parent        # Env 1 "data" root

# --- codebase provenance manifest ------------------------------------------------
prov_path = Path(stage_paths.provenance_manifest)
provenance = json.loads(prov_path.read_text()) if prov_path.exists() else []
if not prov_path.exists():
    (problems if CONFIG["RUN_ACQUISITION"] else warnings_).append(
        f"missing {prov_path.name} (written by the codebase acquisition)")
status_counts = {}
for e in provenance:
    status_counts[e["status"]] = status_counts.get(e["status"], 0) + 1
    if e["status"] in ("CRITICAL",):
        problems.append(f"provenance CRITICAL: {e['product']}: {e['detail'][:200]}")
    elif e["status"] in ("FAIL", "WARN"):
        warnings_.append(f"provenance {e['status']}: {e['product']}: {e['detail'][:160]}")
print("provenance status counts:", status_counts)

# --- codebase coverage audit -------------------------------------------------------
audit_path = Path(stage_paths.active_fires_dir) / "ingestion_audit.json"
audit = json.loads(audit_path.read_text()) if audit_path.exists() else {}
if not audit:
    problems.append("missing active_fires/ingestion_audit.json")
else:
    a = audit.get("audit", {})
    print("coverage audit:", json.dumps(a, indent=1))
    if not audit.get("n_rows"):
        problems.append("normalization produced zero active-fire rows")
    for k, ok in (("chilecito_jul_aug_2026_detections", a.get("chilecito_jul_aug_2026_detections", 0) > 0),
                  ("snpp_ended_by_2026_11_02", a.get("snpp_ended_by_2026_11_02")),
                  ("NOAA-20_through_2026_11_14", a.get("NOAA-20_through_2026_11_14")),
                  ("NOAA-21_through_2026_11_14", a.get("NOAA-21_through_2026_11_14")),
                  ("dza_type2_flares_preserved", a.get("dza_type2_flares_preserved", 0) > 0)):
        if not ok:
            warnings_.append(f"readme 6.3.6 coverage check not met: {k}")

# --- required / expected files -------------------------------------------------------
required = {"active_fires": Path(stage_paths.active_fires_dir),
            "cloud_visibility": Path(stage_paths.cloud_parquet),
            "country_polygons": Path(stage_paths.country_polygons_parquet)}
expected = {"burned_area_validation": Path(stage_paths.burned_area_parquet),
            "cloud_climatology": Path(stage_paths.cloud_parquet).with_name("cloud_climatology.parquet")}
for k, p in required.items():
    if not p.exists():
        problems.append(f"required input missing: {k} ({p})")
for k, p in expected.items():
    if not p.exists():
        warnings_.append(f"optional input missing: {k} ({p.name})")

# --- readability of every parquet --------------------------------------------------
n_rows = {}
for p in sorted(STAGED.rglob("*.parquet")):
    if "staged_raw" in p.parts:
        continue
    try:
        n_rows[str(p.relative_to(STAGED))] = pq.ParquetFile(p).metadata.num_rows
    except Exception as exc:
        problems.append(f"unreadable parquet {p}: {exc}")
af_rows = sum(v for k, v in n_rows.items() if k.startswith("active_fires"))
print(f"active_fires: {sum(1 for k in n_rows if k.startswith('active_fires'))} partitions, {af_rows:,} rows")

if required["country_polygons"].exists():
    isos = set(pq.read_table(required["country_polygons"], columns=["country_iso"])
               .column("country_iso").to_pylist())
    if isos != set(S.TARGET_COUNTRY_ISOS):
        problems.append(f"country polygons {sorted(isos)} != {sorted(S.TARGET_COUNTRY_ISOS)}")
if required["cloud_visibility"].exists():
    cl = pq.read_table(required["cloud_visibility"], columns=["year", "period_kind"]).to_pandas()
    print("cloud grid rows per period_kind:", cl["period_kind"].value_counts().to_dict())
    print("cloud grid years:", sorted(cl["year"].unique().tolist()))
    missing_years = sorted(set(CONFIG["YEARS"]) - set(cl["year"].unique().tolist()))
    if missing_years:
        warnings_.append(f"cloud grid has no rows for years {missing_years} "
                         "(the engine uses the 2012-2025 climatology fallback there)")

# --- preserved originals vs. manifest checksums --------------------------------------
orig_checked = 0
if ORIGINALS_DIR.exists():
    recorded = {}
    for e in provenance:
        try:
            info = json.loads(e.get("detail") or "{}")
        except (ValueError, TypeError):
            continue
        if isinstance(info, dict) and "sha256" in info:
            recorded[info["file"]] = info["sha256"]
    for p in sorted(ORIGINALS_DIR.rglob("*")):
        if p.is_file() and p.name in recorded:
            orig_checked += 1
            if sha256_file(p) != recorded[p.name]:
                problems.append(f"original {p.name} does not match its manifest SHA-256")
print(f"originals verified against manifest: {orig_checked}")

print("\nWARNINGS:\n  " + "\n  ".join(warnings_) if warnings_ else "\nno warnings")
if problems:
    raise RuntimeError("verification failed:\n  " + "\n  ".join(problems))
print("staged data verified.")

## 11. Assemble the offline execution package

```
kaggle_package/
├── package_manifest.json   # contract read by the Kaggle notebook (checksums, commit, layout)
├── dataset-metadata.json   # Kaggle API metadata (written at publish time)
├── codebase/               # `git archive` of the exact commit + REVISION.json
├── staged-data/            # Env 1 outputs: active_fires/, cloud + climatology, burned area,
│                           #   country polygons, data_provenance_manifest.json
├── offline-wheels/         # wheels for the Kaggle interpreter
└── originals/              # (only if PUBLISH_ORIGINALS) untouched downloaded files
```

`staged_raw/` (pre-clip intermediate chunks) is not shipped: `active_fires/` is the
normalized, clipped, deduplicated product the engine consumes.

In [ ]:
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True)


def place(src, dst):
    # Hard-link when possible (no extra disk), copy otherwise.
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


# codebase: exact commit, no .git
cb_dir = PACKAGE_DIR / LAYOUT["codebase"]
cb_dir.mkdir()
archive = subprocess.run(["git", "-C", str(REPO_DIR), "archive", "--format=tar",
                          CODEBASE["commit"]], capture_output=True, check=True).stdout
import io, tarfile
with tarfile.open(fileobj=io.BytesIO(archive)) as tf:
    tf.extractall(cb_dir, filter="data") if sys.version_info >= (3, 12) else tf.extractall(cb_dir)
(cb_dir / "REVISION.json").write_text(json.dumps(CODEBASE, indent=2))

# staged data (Env 1 outputs only)
sd_dir = PACKAGE_DIR / LAYOUT["staged_data"]
for src in sorted(Path(stage_paths.active_fires_dir).rglob("*")):
    if src.is_file():
        place(src, sd_dir / "active_fires" / src.relative_to(stage_paths.active_fires_dir))
for p in list(required.values())[1:] + list(expected.values()) + [prov_path]:
    if Path(p).exists():
        place(Path(p), sd_dir / Path(p).name)

# wheels
for whl in WHEELS_DIR.glob("*.whl"):
    place(whl, PACKAGE_DIR / LAYOUT["wheels"] / whl.name)

# originals (optional; can be very large)
if CONFIG["PUBLISH_ORIGINALS"] and ORIGINALS_DIR.exists():
    for src in ORIGINALS_DIR.rglob("*"):
        if src.is_file():
            place(src, PACKAGE_DIR / LAYOUT["originals"] / src.relative_to(ORIGINALS_DIR))

# checksums of everything we ship
files = {}
total = 0
for p in sorted(PACKAGE_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(PACKAGE_DIR).as_posix()
        files[rel] = {"bytes": p.stat().st_size, "sha256": sha256_file(p)}
        total += files[rel]["bytes"]
content_hash = hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest()
PACKAGE_ID = f"{RUN_ID}-{CODEBASE['commit'][:10]}-{content_hash[:12]}"

manifest = {
    "schema_version": PACKAGE_SCHEMA_VERSION,
    "package_id": PACKAGE_ID,
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "created_by": "notebooks/01_colab_data_acquisition.ipynb",
    "codebase": {**CODEBASE, "path": LAYOUT["codebase"]},
    "layout": {k: v for k, v in LAYOUT.items()
               if (PACKAGE_DIR / v).exists()},
    "target_env2_python": CONFIG["TARGET_ENV2_PYTHON"],
    "years": CONFIG["YEARS"],
    "acquisition": {
        "ran_in_this_package": CONFIG["RUN_ACQUISITION"],
        "probe": PROBE,
        "provenance_status_counts": status_counts,
        "coverage_audit": audit,
        "keep_originals": CONFIG["KEEP_ORIGINALS"],
        "nrt_gap_fill": "FIRMS_MAP_KEY" in DATA_SECRETS,
        "warnings": warnings_,
    },
    "codebase_inspection": {k: INSPECTION[k] for k in
                            ("make_targets", "scripts", "dependency_groups",
                             "requires_python", "env2_expected_inputs", "spec")},
    "wheels": wheels,
    "totals": {"files": len(files), "bytes": total},
    "files": files,
}
(PACKAGE_DIR / MANIFEST_NAME).write_text(json.dumps(manifest, indent=2, default=str))
print(f"package {PACKAGE_ID}: {len(files)} files, {human(total)}")
for name in manifest["layout"].values():
    sz = sum(v["bytes"] for k, v in files.items() if k.startswith(name + "/"))
    print(f"  {name:<16} {human(sz)}")

## 12. Kaggle authentication

Credentials are taken, in order, from: Colab Secrets `KAGGLE_USERNAME` + `KAGGLE_API_TOKEN`
(new-style token); Colab Secrets `KAGGLE_USERNAME` + `KAGGLE_KEY` (legacy key, the `key`
field of `kaggle.json`); an existing `~/.kaggle/kaggle.json`; or an uploaded `kaggle.json`. They are
written only to `~/.kaggle/kaggle.json` with mode `600`, never printed and never placed in
the package. Authentication is validated before any upload.

In [ ]:
KAGGLE_USER = None
if CONFIG["PUBLISH_TO_KAGGLE"]:
    kdir = Path.home() / ".kaggle"
    kdir.mkdir(exist_ok=True)
    kfile = kdir / "kaggle.json"
    user, key = secret("KAGGLE_USERNAME", prompt=False), secret("KAGGLE_KEY", prompt=False)
    token = secret("KAGGLE_API_TOKEN", prompt=False)
    if token:
        # newer Kaggle API tokens (Settings -> API Tokens); username still
        # needed to address <user>/<slug>
        if not user:
            raise RuntimeError("KAGGLE_API_TOKEN given: also set secret KAGGLE_USERNAME")
        tfile = kdir / "access_token"
        tfile.write_text(token)
        os.chmod(tfile, 0o600)
        os.environ["KAGGLE_API_TOKEN"] = token
        kfile.write_text(json.dumps({"username": user, "key": key or ""}))
    elif user and key:
        kfile.write_text(json.dumps({"username": user, "key": key}))
    elif not kfile.exists():
        if not IN_COLAB:
            raise RuntimeError("no Kaggle credentials: set KAGGLE_USERNAME/KAGGLE_KEY or ~/.kaggle/kaggle.json")
        from google.colab import files as colab_files
        print("Upload your kaggle.json (Kaggle -> Settings -> API -> Create New Token)")
        up = colab_files.upload()
        if not up:
            raise RuntimeError("no file uploaded")
        kfile.write_bytes(next(iter(up.values())))
        for name in up:                      # do not leave a copy in the cwd
            Path(name).unlink(missing_ok=True)
    os.chmod(kfile, 0o600)
    KAGGLE_USER = json.loads(kfile.read_text())["username"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    chk = subprocess.run(["kaggle", "datasets", "list", "--mine", "--page-size", "1", "--csv"],
                         capture_output=True, text=True)
    if chk.returncode != 0:
        raise RuntimeError("Kaggle authentication failed: " + (chk.stderr or chk.stdout)[-500:])
    print(f"authenticated to Kaggle as {KAGGLE_USER}")
else:
    print("PUBLISH_TO_KAGGLE=False: skipping authentication")

## 13. Publish (create or new version) and verify

* If `<user>/<KAGGLE_DATASET_SLUG>` does not exist it is **created**; otherwise a **new version**
  is published — never a duplicate dataset.
* Directories are uploaded with `--dir-mode zip`; `--keep-tabular` keeps the Parquet files
  byte-identical (the CLI would otherwise convert tabular files).
* After upload the notebook waits for Kaggle to finish processing, reads back the current
  version number and the file listing, and checks every top-level package entry is present.

In [ ]:
def kaggle_cli(*args, check=True):
    p = subprocess.run(["kaggle", *args], capture_output=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"kaggle {' '.join(args[:3])} failed: {(p.stderr or p.stdout)[-800:]}")
    return p


def dataset_status(ref):
    # {"status": "ready"|"failed"|..., "current_version_number": N}, or None if absent
    p = kaggle_cli("datasets", "status", ref, "--format", "json(status,current_version_number)",
                   check=False)
    txt = p.stdout.strip()
    if p.returncode == 0 and "{" in txt:
        try:
            return json.loads(txt[txt.index("{"):txt.rindex("}") + 1])
        except ValueError:
            pass
    # fallback for CLIs without --format: is the dataset in my list?
    lst = kaggle_cli("datasets", "list", "--mine", "--search", ref.split("/")[1], "--csv",
                     check=False).stdout
    if any(line.split(",")[0] == ref for line in lst.splitlines()[1:]):
        p2 = kaggle_cli("datasets", "status", ref, check=False)
        return {"status": p2.stdout.strip().lower(), "current_version_number": None}
    return None


def is_ready(st):
    return bool(st) and "ready" in str(st.get("status", "")).lower()


PUBLISH = None
if CONFIG["PUBLISH_TO_KAGGLE"]:
    ref = f"{KAGGLE_USER}/{CONFIG['KAGGLE_DATASET_SLUG']}"
    meta = {
        "title": CONFIG["KAGGLE_DATASET_TITLE"],
        "id": ref,
        "licenses": [{"name": "other"}],
        "subtitle": "Offline execution package for the MODIS-VIIRS harmonization engine",
        "description": (
            "Offline execution package produced by notebooks/01_colab_data_acquisition.ipynb.\n\n"
            f"* package_id: `{PACKAGE_ID}`\n* codebase: {CODEBASE['repo_url']} @ "
            f"`{CODEBASE['commit']}`\n\nSources: UMD fire archive (MCD14ML, VNP14IMGML, "
            "VJ114IMGML, VJ214IMGML, MOD14C8Q/MYD14C8Q, Mx14CMQ, MCD64CMQ), NASA FIRMS "
            "(Source Cooperative Portolan mirror, FIRMS API NRT), Natural Earth. NASA/UMD "
            "data are subject to their providers' terms; cite NASA FIRMS and UMD."),
    }
    (PACKAGE_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
    before = dataset_status(ref)
    notes = CONFIG["VERSION_NOTES"] or f"package {PACKAGE_ID} (commit {CODEBASE['commit'][:10]})"
    t0 = time.time()
    if before is None:
        args = ["datasets", "create", "-p", str(PACKAGE_DIR), "-r", "zip", "-t"]
        if CONFIG["KAGGLE_PUBLIC"]:
            args.append("-u")
        action = "created"
    else:
        args = ["datasets", "version", "-p", str(PACKAGE_DIR), "-m", notes, "-r", "zip", "-t"]
        action = "new version"
    r = run(["kaggle", *args], "kaggle_publish")
    st = None
    while time.time() - t0 < CONFIG["PUBLISH_TIMEOUT_MIN"] * 60:
        st = dataset_status(ref)
        if st and "fail" in str(st.get("status", "")).lower():
            raise RuntimeError(f"Kaggle reports the upload as failed: {st}")
        if is_ready(st) and (before is None or st.get("current_version_number") is None
                             or st.get("current_version_number")
                             != before.get("current_version_number")):
            break
        time.sleep(30)
    if not is_ready(st):
        raise RuntimeError(f"dataset not ready after {CONFIG['PUBLISH_TIMEOUT_MIN']} min: {st}")
    listing = kaggle_cli("datasets", "files", ref, "--csv", "--page-size", "200").stdout
    listed = [l.split(",")[0] for l in listing.splitlines()[1:] if l.strip()]
    expected_top = [MANIFEST_NAME] + list(manifest["layout"].values())
    absent = [e for e in expected_top
              if not any(x == e or x.startswith(e + "/") or x.startswith(e + ".") for x in listed)]
    if absent:
        raise RuntimeError(f"published dataset lacks {absent}; listing: {listed[:40]}")
    PUBLISH = {"ref": ref, "action": action, "version": st.get("current_version_number"),
               "package_id": PACKAGE_ID, "commit": CODEBASE["commit"],
               "seconds": round(time.time() - t0), "listing_sample": listed[:40]}
    (WORK / f"publish_record_{RUN_ID}.json").write_text(json.dumps(PUBLISH, indent=2))
    print(json.dumps(PUBLISH, indent=2))
else:
    print(f"PUBLISH_TO_KAGGLE=False: package ready at {PACKAGE_DIR} (not uploaded)")

## 14. Summary and next step

In the Kaggle notebook (`02_kaggle_harmonization.ipynb`): **Add Input → Your Datasets →**
the dataset above (or *Check for updates* to switch to the new version) and run it. It
locates `package_manifest.json`, verifies the checksums, and runs this exact commit offline —
no edits are needed when a new version is published.

In [ ]:
summary = {
    "package_id": PACKAGE_ID,
    "codebase_commit": CODEBASE["commit"],
    "files": manifest["totals"]["files"],
    "size": human(manifest["totals"]["bytes"]),
    "acquisition_ran": CONFIG["RUN_ACQUISITION"],
    "warnings": len(warnings_),
    "kaggle": PUBLISH or "not published",
}
print(json.dumps(summary, indent=2))